# 89 — Campaign B mass autonomous exploration

できるだけ多くの S2 候補を自動探索する。

- 複数 wave（基本空間 → 拡張空間）
- 既出 `normalized_scheme_key` はスキップ
- 共有 M2 欠落時は auto-generate（j2>=2 staged）
- `never_stop`：キュー消化まで継続
- 全成果物 `NOT_CERTIFIED` / `SCREENING_ONLY`（production M6 禁止）


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'mass_explore.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/mass_explore.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M2_SPLIT_BATCH_TO', '1')
os.environ.setdefault('VALIDATED_RG_CHECKPOINT_KEEP', '5')
os.environ.setdefault('VALIDATED_RG_M2_ALLOW_CODE_DRIFT', '1')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.mass_explore import run_mass_explore
import shutil

validate_persistent_root()
BASE = PROJECT_ROOT / 'configs' / 'campaign_b_mass_explore.yaml'
assert BASE.is_file(), BASE
RUNTIME = PERSIST_ROOT / 'campaign_b' / '_mass_explore' / 'runtime'
RUNTIME.mkdir(parents=True, exist_ok=True)
CFG = RUNTIME / 'mass_explore_run.yaml'
text = BASE.read_text(encoding='utf-8')
lines = []
for line in text.splitlines():
    if line.startswith('persistent_root:'):
        lines.append(f'persistent_root: {PERSIST_ROOT}')
    else:
        lines.append(line)
CFG.write_text('\n'.join(lines) + '\n', encoding='utf-8')
# Ensure space YAMLs are next to runtime config copies used by waves
for name in ('campaign_b_s2_space_v1.yaml', 'campaign_b_s2_space_expanded_v1.yaml'):
    src = PROJECT_ROOT / 'configs' / name
    if src.is_file():
        shutil.copy2(src, RUNTIME / name)
        # mass_explore resolves spaces relative to config parent → also copy beside CFG parent
        shutil.copy2(src, CFG.parent / name)
print('CFG', CFG)


In [ ]:
SESSION = run_mass_explore(CFG)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'status': SESSION.get('status'),
    'selected_total': SESSION.get('selected_total'),
    'archived_total': SESSION.get('archived_total'),
    'processed_scheme_keys': SESSION.get('processed_scheme_keys'),
    'waves': [
        {
            'wave': w.get('wave'),
            'space': w.get('space'),
            'campaign_run_id': w.get('campaign_run_id'),
            'terminal_reason': w.get('terminal_reason'),
            'selected_count': w.get('selected_count'),
            'new_schemes': w.get('new_schemes'),
        }
        for w in (SESSION.get('waves') or [])
    ],
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
}, indent=2, ensure_ascii=False, default=str))
